# 14 — LPPLS + XGBoost: wykrywanie stanów anomalii spekulacyjnej
Cel: klasyfikować w dniu t stan wg referencji BSADF z notatnika 13.
To **naśladowanie niezależnego testu referencyjnego / nowcasting**, a nie
bezpośrednie wykrywanie fundamentalnej bańki ani przewidywanie jej końca.

Proponowany model: XGBoost (Price+LPPLS). Benchmarki: XGB Price-only,
XGB LPPLS-only, regresja logistyczna, prosta reguła Confidence Indicator.

Outer: rozszerzający się trening do roku t-2, walidacja t-1, test t.
Inner: min. dwa podziały chronologiczne tylko w treningu do doboru HP.
Próg dobierany tylko na walidacji t-1; test nie wpływa na wybór.
Wszystkie modele są trenowane na tych samych datach.

In [ ]:
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.model_selection import ParameterSampler
from xgboost import XGBClassifier
import joblib
import sys
sys.path.insert(0, str(Path('.').resolve() / 'src'))
from bubble_tools import load_prices, classification_stats
from lppls_research import build_lppls_daily_features, TemporalAvailabilityValidator
from lppls_research.features import LPPLS_METRICS

ROOT=Path('.')
OUT=ROOT/'14_wyniki_bubble'
OUT.mkdir(exist_ok=True,parents=True)
REF_FILE=ROOT/'13_wyniki_bubble'/'13_bubble_reference_daily.csv'
FITS_FILE=ROOT/'lppls_rolling_results.csv'
if not REF_FILE.exists():raise FileNotFoundError('Uruchom najpierw notatnik 13')
if not FITS_FILE.is_file():raise FileNotFoundError(f'Brak pełnych dopasowań LPPLS: {FITS_FILE}')

TEST_YEARS=[2021,2022,2023,2024]
N_SEARCH=12  # do pilota 3; do badania: 20–40, bez wyboru na podstawie testu
SEED=42
THRESHOLDS=np.round(np.arange(.05,.951,.05),2)
MAX_TIME_IN_ALARM=0.30 # predefiniowany cel operacyjny; nie jest standardem finansowym
MODELS=['XGBoost','LogisticRegression'] # podstawowe; benchmark CI dodany osobno
FEATURE_GROUPS_TO_RUN=['Price-only','LPPLS-only','Price+LPPLS']

## 1. Cechy dzienne BTC (dostępne najpóźniej w dniu t)
Zbiór referencyjny 13 zawiera jedynie **stany as-of-day** BSADF.
Nie przekazujemy modelom BSADF score, UNION, dat ani przyszłych spadków.

In [ ]:
ref=pd.read_csv(REF_FILE,parse_dates=['date','reference_available_at'])
prices=load_prices(REF_FILE)
TemporalAvailabilityValidator.check_calendar(prices, require_consecutive_days=True)
fits=pd.read_csv(FITS_FILE, parse_dates=['win_start','win_end'])
TemporalAvailabilityValidator.check_windows(fits)
assert len(fits)==107_240, f'Expected 107240 BTC LPPLS windows, got {len(fits)}'
lppls_daily=build_lppls_daily_features(fits,calendar=prices['date'])

# Retain the original price-only feature definitions from bubble_tools.
features=prices[['date','price']].copy().sort_values('date').reset_index(drop=True)
logp=np.log(features.price)
features['ret_1d']=logp.diff()
for k in (7,14,30,60):
    features[f'momentum_{k}']=logp.diff(k)
for k in (7,30,60):
    features[f'vol_{k}']=features.ret_1d.rolling(k,min_periods=k).std()
for k in (30,90):
    features[f'drawdown_{k}']=features.price/features.price.rolling(k,min_periods=k).max()-1
features=features.merge(lppls_daily,on='date',how='left',validate='one_to_one')
price_cols=['ret_1d']+[f'momentum_{k}' for k in (7,14,30,60)] + \
           [f'vol_{k}' for k in (7,30,60)] + [f'drawdown_{k}' for k in (30,90)]
lppls_cols=['lppls_ci','lppls_n_accepted']+list(LPPLS_METRICS)
groups={'Price-only':price_cols, 'LPPLS-only':lppls_cols,
        'Price+LPPLS':price_cols+lppls_cols}

# If notebook 08b has been run, require identical counts and CI.
comparison=ROOT/'08b_wyniki_tuning'/'01_cechy_dzienne.csv'
if comparison.exists():
    previous=pd.read_csv(comparison,parse_dates=['date'])
    TemporalAvailabilityValidator.check_matching_counts(
        previous,lppls_daily,mapping={'n_all':'lppls_n_all',
                                      'lppls_n_accepted':'lppls_n_accepted',
                                      'lppls_ci':'lppls_ci'})

data=features.merge(ref[['date','bubble_state_gsadf','bubble_union','reference_available_at']],
                    on='date',how='inner',validate='one_to_one')
TemporalAvailabilityValidator.check_as_of(
    data,available_col='reference_available_at',allow_missing=True
)
assert data.date.is_unique
data=data.loc[data.bubble_state_gsadf.notna()].copy().sort_values('date').reset_index(drop=True)
data['y']=data.bubble_state_gsadf.astype(int)
data.to_csv(OUT/'14_daily_model_data.csv',index=False)
print('All fits:',int(lppls_daily.lppls_n_all.sum()),
      'accepted fits:',int(lppls_daily.lppls_n_accepted.sum()))
print('Sample with reference:',len(data),'positive:',int(data.y.sum()))
print('Feature groups:',{k:len(v) for k,v in groups.items()})


## 2. Klasyfikatory, wewnętrzne strojenie i wybór progu
Używamy imputacji/scalera uczonych WYŁĄCZNIE na części treningowej.

In [ ]:
SEARCH_SPACES={
    'XGBoost':{'n_estimators':[100,180,300], 'max_depth':[2,3,4],
               'learning_rate':[.03,.07,.12], 'subsample':[.8,1.0],
               'colsample_bytree':[.8,1.0], 'min_child_weight':[1,5],
               'reg_lambda':[1,5,15]},
    'LogisticRegression':{'C':[.01,.1,.3,1,3,10], 'class_weight':[None,'balanced']},
}


def make_estimator(name,params):
    if name=='XGBoost':
        model=XGBClassifier(objective='binary:logistic',eval_metric='logloss',
                            tree_method='hist',n_jobs=1,random_state=SEED,**params)
        return make_pipeline(SimpleImputer(strategy='constant',fill_value=0),model)
    if name=='LogisticRegression':
        model=LogisticRegression(max_iter=1200,random_state=SEED,**params)
        return make_pipeline(SimpleImputer(strategy='constant',fill_value=0),
                             StandardScaler(),model)
    raise ValueError(name)


def safe_ap(y,s):
    return float(average_precision_score(y,s)) if len(np.unique(y))==2 else np.nan


def choose_hyperparams(name,feat_cols,train):
    candidates=list(ParameterSampler(SEARCH_SPACES[name],n_iter=N_SEARCH,random_state=SEED))
    train_years=sorted(train.date.dt.year.unique())
    # Ostatnie maks. 3 lata treningowe jako wewnętrzne okresy walidacyjne.
    val_years=train_years[-3:]
    allrows=[]
    for c_id,params in enumerate(candidates):
        aps=[]
        for v in val_years:
            a=train[train.date.dt.year<v]
            b=train[train.date.dt.year==v]
            if a.date.dt.year.nunique()<2 or a.y.nunique()<2 or b.y.nunique()<2:
                continue
            estimator=make_estimator(name,params)
            estimator.fit(a[feat_cols],a.y)
            aps.append(safe_ap(b.y,estimator.predict_proba(b[feat_cols])[:,1]))
        score=float(np.nanmean(aps)) if aps else np.nan
        allrows.append({'candidate':c_id,'model':name,'params':json.dumps(params),
                        'mean_inner_AP':score,'inner_folds':len(aps)})
    search=pd.DataFrame(allrows)
    valid=search[search.mean_inner_AP.notna()]
    if valid.empty:
        raise ValueError(f'Brak poprawnych wewnętrznych foldów dla {name}. Zbadaj liczbę dodatnich epizodów.')
    bestrow=valid.sort_values(['mean_inner_AP','candidate'],ascending=[False,True]).iloc[0]
    return json.loads(bestrow.params),search


def choose_threshold(y,s):
    """Progi na walidacji; ograniczenie Time in Alarm, następnie balanced accuracy."""
    y=np.asarray(y,dtype=int);s=np.asarray(s,float)
    rows=[]
    for t in THRESHOLDS:
        pred=(s>=t).astype(int)
        m=classification_stats(y,pred)
        rows.append({'threshold':float(t),'time_in_alarm':m['time_in_alarm'],
                     'balanced_accuracy':m['balanced_accuracy'],
                     'specificity':m['specificity'],'recall':m['recall']})
    tab=pd.DataFrame(rows)
    eligible=tab[tab.time_in_alarm<=MAX_TIME_IN_ALARM]
    if y.sum()==0 or y.sum()==len(y):
        # Nie próbuj maksymalizować metryki o nieokreślonej klasie.
        return 0.5,'reference_no_both_classes',tab
    if eligible.empty:
        return 0.5,'no_threshold_meets_alarm_budget',tab
    choice=eligible.sort_values(['balanced_accuracy','time_in_alarm','threshold'],
                                ascending=[False,True,True]).iloc[0]
    return float(choice.threshold),'validated_budget',tab

## 3. Walk-forward OOS 2021–2024 (bez używania testu do wyboru modelu)

In [ ]:
fold_rows=[];pred_rows=[];search_rows=[];threshold_rows=[]
for year in TEST_YEARS:
    train=data[data.date.dt.year<=year-2].copy()
    val=data[data.date.dt.year==year-1].copy()
    test=data[data.date.dt.year==year].copy()
    if train.empty or val.empty or test.empty:raise ValueError(f'Brakuje danych dla testu {year}')
    TemporalAvailabilityValidator.check_split(train,val,test)
    if train.y.nunique()<2:
        raise ValueError(f'Trening {year} ma tylko jedną klasę referencyjną BSADF; nie da się wytrenować XGBoost.')
    for model_name in MODELS:
        for group_name in FEATURE_GROUPS_TO_RUN:
            cols=groups[group_name]
            params,history=choose_hyperparams(model_name,cols,train)
            history['test_year']=year;history['feature_group']=group_name
            search_rows.append(history)
            model=make_estimator(model_name,params)
            model.fit(train[cols],train.y)
            score_val=model.predict_proba(val[cols])[:,1]
            threshold,status,curve=choose_threshold(val.y,score_val)
            curve['year']=year;curve['model']=model_name;curve['feature_group']=group_name
            threshold_rows.append(curve)
            score_test=model.predict_proba(test[cols])[:,1]
            ypred=(score_test>=threshold).astype(int)
            stats=classification_stats(test.y,ypred,score_test)
            fold_rows.append({'year':year,'model':model_name,'feature_group':group_name,
                              'threshold':threshold,'selection_status':status,
                              'best_params':json.dumps(params),**stats})
            p=test[['date','price','y','bubble_union']].copy()
            p['year']=year;p['model']=model_name;p['feature_group']=group_name
            p['score']=score_test;p['y_pred']=ypred;p['threshold']=threshold
            pred_rows.append(p)
    # Trivial benchmark evaluated on the very same test days.
    baseline=test[['date','price','y','bubble_union']].copy()
    baseline['year']=year
    baseline['model']='AlwaysPositive'
    baseline['feature_group']='Baseline'
    baseline['score']=1.0
    baseline['y_pred']=1
    baseline['threshold']=0.0
    pred_rows.append(baseline)
    baseline_stats=classification_stats(test.y,baseline.y_pred,baseline.score)
    fold_rows.append({'year':year,'model':'AlwaysPositive',
        'feature_group':'Baseline','threshold':0.0,
        'selection_status':'fixed_reference',
        'best_params':'{}',**baseline_stats})
    # Niewytrenowana reguła CI: historyczna reguła alarmowa; nie używa GSADF w treningu.
    for th in [0.,.05]:
        p=test[['date','price','y','bubble_union']].copy()
        p['year']=year;p['model']=f'CI_gt_{th:g}';p['feature_group']='LPPLS-rule'
        p['score']=test.lppls_ci.to_numpy();p['y_pred']=(test.lppls_ci>th).astype(int)
        p['threshold']=th
        pred_rows.append(p)
        stats=classification_stats(test.y,p.y_pred,p.score)
        fold_rows.append({'year':year,'model':f'CI_gt_{th:g}',
                          'feature_group':'LPPLS-rule','threshold':th,
                          'selection_status':'fixed_reference','best_params':'{}',**stats})

folds=pd.DataFrame(fold_rows)
preds=pd.concat(pred_rows,ignore_index=True)
folds.to_csv(OUT/'14_bubble_detection_model_results.csv',index=False)
preds.to_csv(OUT/'14_bubble_detection_predictions_OOS.csv',index=False)
pd.concat(search_rows,ignore_index=True).to_csv(OUT/'14_hyperparameter_search.csv',index=False)
pd.concat(threshold_rows,ignore_index=True).to_csv(OUT/'14_validation_threshold_curves.csv',index=False)
display(folds[['year','model','feature_group','positives','AP','ROC_AUC','f1','time_in_alarm']].round(3))

## 4. Podsumowanie rocznych i połączonych prognoz OOS
AP dla całej próby obliczamy z prawdopodobieństw, nie ze średniej AP rocznej.

In [ ]:
summary=[]
for (model,group),p in preds.groupby(['model','feature_group']):
    if not p.date.is_unique:raise ValueError('Duplikaty dni w OOS')
    stats=classification_stats(p.y,p.y_pred,p.score)
    summary.append({'model':model,'feature_group':group,**stats})
summary=pd.DataFrame(summary).sort_values('AP',ascending=False)
summary.to_csv(OUT/'14_bubble_detection_summary.csv',index=False)
display(summary.round(3))

In [ ]:
fig,ax=plt.subplots(figsize=(11,5))
heat=summary.loc[summary.model.isin(MODELS)].pivot(index='model',columns='feature_group',values='AP')
heat.plot.bar(ax=ax,rot=0)
ax.set(title='Detekcja stanów BSADF: AP na OOS',ylabel='Average Precision',xlabel='')
ax.grid(axis='y',alpha=.25)
fig.tight_layout();fig.savefig(OUT/'14_01_AP_OOS.png',dpi=170);plt.show()

## 5. Zamrożony model BTC do walidacji międzyrynkowej (notatnik 16)
Trening do 2019, wybór progu z 2020; NIE dopasowujemy go do innego rynku.
Model służy do transferu, nie jest modelem dobranym po wynikach 2021–24.

In [ ]:
transfer_train=data[data.date.dt.year<=2019]
transfer_val=data[data.date.dt.year==2020]
if transfer_train.y.nunique()==2 and not transfer_val.empty:
    cols=groups['Price+LPPLS']
    params,_=choose_hyperparams('XGBoost',cols,transfer_train)
    transfer_model=make_estimator('XGBoost',params)
    transfer_model.fit(transfer_train[cols],transfer_train.y)
    tscore=transfer_model.predict_proba(transfer_val[cols])[:,1]
    tthreshold,tstatus,_=choose_threshold(transfer_val.y,tscore)
    bundle={'model':transfer_model,'feature_cols':cols,'threshold':float(tthreshold),
            'threshold_status':tstatus,'model_name':'XGBoost',
            'feature_group':'Price+LPPLS','train_end':'2019-12-31',
            'calibration_year':2020,'label':'BSADF_sampled_asof_state'}
    joblib.dump(bundle,OUT/'14_transfer_BTC_XGB_PriceLPPLS.joblib')
    print('Zapisano model transferowy. Threshold:',tthreshold,'status:',tstatus)
else:
    print('Brak obu klas w treningu 2015–2019 albo brakuje 2020. Transfer nie został zapisany.')

print('Wyniki:',OUT.resolve())